# Synora · Intentions d'achat sur un site marchand (sujet 18)

**Équipe Synora** · Machine Learning, L3 GLSI, IIT Sfax, 2026-2027.
Repérer, **pendant la visite**, les sessions d'un site marchand qui vont se terminer par un achat (Online Shoppers Purchasing Intention Dataset, UCI).

Ce notebook unique suit les **six étapes du cycle de vie d'un modèle**. Exécutez-le de haut en bas (*Exécution → Tout exécuter*).

| Étape | Partie du notebook | Jalon |
| --- | --- | --- |
| 1 · Collecte | Partie 1 : charger les données officielles, contrôler leur qualité, explorer | J1 |
| 2 · Préparation | Partie 2 : nettoyer, séparer train / test **avant** de préparer, encoder et normaliser | J1 |
| 3 · Modèle | Partie 3 : référence naïve et modèles candidats | J2 |
| 4 · Entraînement | Partie 4 : validation croisée avec et sans `PageValues`, meilleur seuil, `GridSearchCV` | J2 |
| 5 · Évaluation | Partie 5 : choix du seuil, évaluation **unique** sur le test, analyse des erreurs | J3 |
| 6 · Déploiement | Partie 6 : export de `model.joblib` et `requirements.txt` pour l'application Streamlit | J3 |

**Règle d'or :** le jeu de test (20 %) est mis de côté en partie 2 et n'est utilisé qu'**une seule fois**, en partie 5. Ce n'est pas une ligne droite : une évaluation décevante renvoie à la préparation des données.

In [ ]:
# Imports (toutes ces bibliothèques sont déjà installées dans Colab)
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## 0. Mise en place

In [ ]:
# ===================== CONFIG : tous les réglages du projet au même endroit =====================
RANDOM_STATE = 42
TEST_SIZE = 0.2
CV_FOLDS = 5

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00468/online_shoppers_intention.csv"
LOCAL_PATH = "online_shoppers_intention.csv"   # utilisé si le fichier a été téléversé dans Colab

TARGET = "Revenue"
NUM_COLS = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "SpecialDay",
]
LEAKY_COL = "PageValues"   # calculée à partir des transactions : testée avec et sans
CAT_COLS = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
BOOL_COLS = ["Weekend"]

FEATURE_SETS = {
    "sans_PageValues": NUM_COLS + CAT_COLS + BOOL_COLS,
    "avec_PageValues": NUM_COLS + [LEAKY_COL] + CAT_COLS + BOOL_COLS,
}

# Garde-fou : le découpage doit toujours être le même (reproductibilité).
EXPECTED_SPLIT = (9764, 2441)   # (lignes d'entraînement, lignes de test)

In [ ]:
# ===================== FONCTIONS DU PROJET =====================
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def load_raw():
    """Étape 1 : charge le CSV officiel (fichier local, sinon page UCI, sinon téléversement)."""
    if Path(LOCAL_PATH).exists():
        return pd.read_csv(LOCAL_PATH)
    try:
        return pd.read_csv(DATA_URL)
    except Exception as err:
        print("Téléchargement impossible :", err, "-> téléversez online_shoppers_intention.csv")
        from google.colab import files
        files.upload()
        return pd.read_csv(LOCAL_PATH)


def clean(df):
    """Nettoyage sans apprentissage (aucune statistique calculée) : sûr avant le découpage."""
    df = df.copy()
    df["Month"] = df["Month"].replace({"June": "Jun"})   # seul mois écrit en entier
    df["Weekend"] = df["Weekend"].astype(int)
    df[TARGET] = df[TARGET].astype(int)
    for col in CAT_COLS:                                 # les codes sont des catégories
        df[col] = df[col].astype(str)
    return df.drop_duplicates().reset_index(drop=True)   # avant le découpage, sinon fuite


def split(df):
    """Découpage stratifié, toujours avec la même graine."""
    X = df.drop(columns=TARGET)
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
    assert (len(X_train), len(X_test)) == EXPECTED_SPLIT, "Découpage différent de celui attendu !"
    return X_train, X_test, y_train, y_test


def make_preprocessor(feature_set):
    """Étape 2 : à n'ajuster que sur l'entraînement (ou dans un Pipeline)."""
    num = [c for c in FEATURE_SETS[feature_set] if c in NUM_COLS + [LEAKY_COL]]
    return ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
        ("bool", "passthrough", BOOL_COLS),
    ], remainder="drop")   # remainder="drop" retire PageValues dans la variante « sans »


def build_estimator(name, params=None):
    """Étape 3 : les modèles candidats."""
    estimator = {
        "Reference_naive": DummyClassifier(strategy="most_frequent"),
        "Regression_logistique": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "Foret_aleatoire": RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "Gradient_boosting": HistGradientBoostingClassifier(
            class_weight="balanced", random_state=RANDOM_STATE),
    }[name]
    if params:
        estimator.set_params(**params)
    return estimator


def make_pipeline(name, feature_set, params=None):
    return Pipeline([("prep", make_preprocessor(feature_set)),
                     ("model", build_estimator(name, params))])


CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Partie 1 · Collecte (étape 1)

### 1.1 Premier contact

In [ ]:
raw = load_raw()
print(raw.shape)
raw.head()

### 1.2 Qualité : types, valeurs manquantes, doublons

In [ ]:
raw.info()
print("\nValeurs manquantes :", int(raw.isna().sum().sum()))
print("Lignes en double   :", int(raw.duplicated().sum()))
raw.describe().T

#### 1.2 bis Les doublons : erreurs ou vraies visites ?

Avant de retirer les 125 doublons, regardons à quoi ils ressemblent.

In [ ]:
dups = raw[raw.duplicated(keep=False)]
print("Lignes concernées par un doublon :", len(dups))
print("Dont achats :", int(dups[TARGET].sum()))
print("Pages produits (médiane) :", dups["ProductRelated"].median(),
      "| durée produits (médiane) :", dups["ProductRelated_Duration"].median(), "s")
print("Taux de rebond = 0,2 (rebond maximal) :", int((dups["BounceRates"] == 0.2).sum()), "sur", len(dups))

# Mêmes variables mais étiquette différente ? (ce serait un vrai problème de qualité)
features = [c for c in raw.columns if c != TARGET]
conflits = (raw.groupby(features)[TARGET].nunique() > 1).sum()
print("Groupes identiques avec des étiquettes contradictoires :", int(conflits))

#### 1.2 ter Valeurs aberrantes et durées nulles

Les durées sont très étirées vers la droite : une échelle logarithmique (`log1p`) permet de voir toute la distribution.

In [ ]:
DUR_COLS = ["Administrative_Duration", "Informational_Duration", "ProductRelated_Duration"]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, col in zip(axes, DUR_COLS):
    sns.histplot(np.log1p(raw[col]), bins=40, ax=ax)
    ax.set_title(col)
    ax.set_xlabel("log(1 + durée en secondes)")
plt.tight_layout()
plt.show()

print("Asymétrie (skew) :", raw[DUR_COLS].skew().round(1).to_dict())
pr = raw["ProductRelated_Duration"]
print(f"Pages produits : médiane {pr.median() / 60:.0f} min | 99e centile {pr.quantile(0.99) / 3600:.1f} h | max {pr.max() / 3600:.1f} h")
print("Sessions de plus d'1 h sur les pages produits :", int((pr > 3600).sum()))
print("Sessions sans aucune page vue :", int((raw[["Administrative", "Informational", "ProductRelated"]].sum(axis=1) == 0).sum()))

# Google Analytics ne mesure pas la durée de la dernière page vue : des pages > 0 avec une durée = 0 sont normales
for pages, duree in [("Administrative", "Administrative_Duration"), ("Informational", "Informational_Duration"),
                     ("ProductRelated", "ProductRelated_Duration")]:
    print(f"{pages:15s} pages > 0 mais durée = 0 : {int(((raw[pages] > 0) & (raw[duree] == 0)).sum()):4d}"
          f" | pages = 0 mais durée > 0 : {int(((raw[pages] == 0) & (raw[duree] > 0)).sum())}")

### 1.3 La cible : classes déséquilibrées

In [ ]:
print(raw[TARGET].value_counts())
print(raw[TARGET].value_counts(normalize=True).round(3))
sns.countplot(x=TARGET, data=raw)
plt.title("Répartition de Revenue (achat ou non)")
plt.show()

### 1.4 Les catégories : mois et type de visiteur

In [ ]:
print(raw["Month"].value_counts())       # remarque : 'June' en entier, pas de janvier ni d'avril
print(raw["VisitorType"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
sns.barplot(x="Month", y=TARGET, data=raw, order=order, ax=axes[0])
axes[0].set_title("Taux d'achat par mois")
sns.barplot(x="VisitorType", y=TARGET, data=raw, ax=axes[1])
axes[1].set_title("Taux d'achat par type de visiteur")
plt.tight_layout()
plt.show()

### 1.5 Le piège `PageValues`

In [ ]:
has_value = raw[LEAKY_COL] > 0
print("Taux d'achat quand PageValues > 0 :", round(raw.loc[has_value, TARGET].mean(), 3),
      "(", int(has_value.sum()), "sessions )")
print("Taux d'achat quand PageValues = 0 :", round(raw.loc[~has_value, TARGET].mean(), 3),
      "(", int((~has_value).sum()), "sessions )")

In [ ]:
num = NUM_COLS + [LEAKY_COL]
plt.figure(figsize=(9, 7))
sns.heatmap(raw[num + [TARGET]].astype(float).corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Corrélations des variables numériques avec la cible")
plt.show()

### 1.5 bis Qu'est-ce qui distingue les acheteurs ?

Comparaison acheteurs / non-acheteurs sur les variables **connues pendant la visite** (donc sans `PageValues`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(x=TARGET, y="ExitRates", data=raw, ax=axes[0])
sns.boxplot(x=TARGET, y="ProductRelated", data=raw, ax=axes[1])
axes[1].set_yscale("symlog")
sns.boxplot(x=TARGET, y=np.log1p(raw["ProductRelated_Duration"]), data=raw, ax=axes[2])
axes[2].set_ylabel("log(1 + ProductRelated_Duration)")
for ax in axes:
    ax.set_xticks([0, 1], ["Pas d'achat", "Achat"])
    ax.set_xlabel("")
plt.suptitle("Acheteurs et non-acheteurs")
plt.tight_layout()
plt.show()

print(raw.groupby(TARGET)[["ExitRates", "BounceRates", "ProductRelated", "ProductRelated_Duration"]].median().round(3))

# Corrélation de Spearman (sur les rangs) : plus fiable que Pearson pour des variables très asymétriques
print("\nCorrélation de Spearman avec l'achat :")
print(raw[NUM_COLS + [LEAKY_COL]].corrwith(raw[TARGET].astype(int), method="spearman").sort_values().round(2))

### 1.5 ter Variables à surveiller : `SpecialDay` et codes rares

In [ ]:
print("SpecialDay > 0 seulement pour les mois :", sorted(raw.loc[raw["SpecialDay"] > 0, "Month"].unique()))
print("(proximité de la Saint-Valentin en février et de la fête des mères en mai : redondant en partie avec Month)\n")

for col in ["OperatingSystems", "Browser", "Region", "TrafficType"]:
    counts = raw[col].value_counts()
    rares = counts[counts < 30]
    print(f"{col:16s} {len(counts):2d} codes, dont {len(rares)} rares (< 30 sessions) : {rares.to_dict()}")

### 1.6 Constats

- **Qualité des données :** il n'y a aucune valeur manquante. On trouve 125 doublons, mais en les regardant (1.2 bis), ce sont tous des rebonds : une seule page vue, 0 seconde, pas d'achat. Ce sont donc sûrement des visites différentes qui se ressemblent, pas des erreurs de saisie.
- **Durées :** elles sont très étalées. La moitié des visites passent moins de 10 minutes sur les pages produits, mais 867 visites dépassent 1 heure et la plus longue dure presque 18 heures (sans doute un onglet oublié). On les garde. On voit aussi 717 visites avec des pages produits vues mais une durée de 0 : c'est normal, car Google Analytics ne compte pas le temps passé sur la dernière page.
- **Déséquilibre :** seulement 1 908 visites sur 12 330 finissent par un achat (15,5 %). Un modèle qui dirait toujours « pas d'achat » aurait 84,5 % de bonnes réponses sans trouver un seul acheteur. C'est pour ça qu'on regardera le rappel et le F1 sur les achats, pas l'exactitude.
- **`PageValues` :** quand elle est supérieure à 0, 56 % des visites achètent ; quand elle vaut 0, seulement 4 %. C'est la variable la plus liée à l'achat (0,63 en Spearman). Le problème : elle est calculée avec les achats, donc on ne la connaît pas vraiment pendant la visite. On fera les modèles avec et sans elle pour voir la différence.
- **Ce qui distingue les acheteurs (1.5 bis) :** ils regardent presque deux fois plus de pages produits (29 contre 16 en médiane), y restent plus longtemps et quittent moins vite le site (`ExitRates` : −0,25).
- **Mois et visiteurs :** il manque janvier et avril, et « June » était écrit en entier. D'après les graphiques de la section 1.4, on achète beaucoup plus en novembre qu'en février, et les nouveaux visiteurs achètent plus souvent que les habitués. `SpecialDay` n'apparaît qu'en février et en mai, donc elle répète en partie l'information du mois.
- **Codes en nombres :** `OperatingSystems`, `Browser`, `Region` et `TrafficType` sont des codes, pas des quantités : la région 8 n'est pas « plus » que la région 2. On les encode en one-hot. 12 codes ont moins de 30 visites ; on les garde pour l'instant.

# Partie 2 · Préparation (étape 2)

### 2.1 Nettoyage sans apprentissage

**Pourquoi retirer les doublons alors que ce sont probablement de vraies visites (section 1.2 bis) ?** Une même ligne pourrait se retrouver à la fois dans l'entraînement et dans le test : le modèle serait alors évalué sur des exemples déjà vus (fuite). Les 125 lignes retirées sont toutes des rebonds sans achat ; l'effet sur la répartition est négligeable (15,5 % → 15,6 % d'achats). Choix prudent, assumé.

In [ ]:
df = clean(raw)
print("Lignes avant :", len(raw), "| après suppression des doublons :", len(df))
print(sorted(df["Month"].unique()))

### 2.2 Séparer **avant** de préparer

Le découpage est stratifié pour garder environ 15 % d'achats dans le train **et** dans le test.

In [ ]:
X_train, X_test, y_train, y_test = split(df)
print("Entraînement :", X_train.shape, "| Test :", X_test.shape)
print("Taux d'achat train :", round(y_train.mean(), 3), "| test :", round(y_test.mean(), 3))

### 2.3 Encoder et normaliser avec un `ColumnTransformer`

`fit` sur l'entraînement uniquement ; le test est seulement **transformé**.

In [ ]:
for name in FEATURE_SETS:
    prep = make_preprocessor(name)
    Xtr = prep.fit_transform(X_train)
    Xte = prep.transform(X_test)
    print(f"{name:16s} -> train {Xtr.shape} | test {Xte.shape}")

prep = make_preprocessor("sans_PageValues").fit(X_train)
print("\nExemples de colonnes créées :", list(prep.get_feature_names_out()[:12]))

### 2.4 Préparer une nouvelle session (comme le fera l'application)

In [ ]:
nouvelle_session = X_test.iloc[[0]]
display(nouvelle_session)
print(prep.transform(nouvelle_session).round(2))

## Bilan J1

- **Ce qu'on a appris :** les données viennent d'un seul site sur un an (sans janvier ni avril). Elles sont propres mais déséquilibrées : 15,5 % d'achats. Les acheteurs regardent plus de pages produits et partent moins vite. `PageValues` prédit très bien l'achat, mais elle pose un risque de fuite.
- **Nos choix de préparation :** on a retiré les doublons avant de découper, pour qu'une même ligne ne se retrouve pas à la fois dans l'entraînement et dans le test (ça change très peu : 15,5 % → 15,6 % d'achats). On a découpé en 80 / 20 en gardant la même proportion d'achats (`stratify`, graine 42), **avant** de préparer. Ensuite, `StandardScaler` pour les nombres et `OneHotEncoder` pour les codes, appris uniquement sur l'entraînement. On obtient 9 764 visites pour entraîner et 2 441 pour tester ; le test reste de côté jusqu'à la partie 5.
- **Pour la suite (J2) :** comparer les modèles avec et sans `PageValues`.

# Partie 3 · Modèle (étape 3)

| Modèle | Pourquoi l'essayer |
| --- | --- |
| Référence naïve | Prédit toujours « pas d'achat » : le score à battre |
| Régression logistique | Simple, rapide, interprétable |
| Forêt aléatoire | Capte les interactions sans normalisation |
| Gradient boosting | Souvent le plus performant sur des données tabulaires |

Tous les modèles (sauf la référence) utilisent `class_weight="balanced"` à cause des 15 % d'achats.

In [ ]:
MODELS = ["Reference_naive", "Regression_logistique", "Foret_aleatoire", "Gradient_boosting"]
SCORING = {"f1": "f1", "rappel": "recall", "precision": "precision", "auc": "roc_auc"}

# Partie 4 · Entraînement (étape 4)

### 4.1 Validation croisée de tous les modèles, avec et sans `PageValues`

In [ ]:
from sklearn.model_selection import cross_validate

rows = []
for feature_set in FEATURE_SETS:
    for name in MODELS:
        scores = cross_validate(make_pipeline(name, feature_set), X_train, y_train,
                                cv=CV, scoring=SCORING, n_jobs=-1)
        row = {"variables": feature_set, "modele": name}
        for metric in SCORING:
            values = scores[f"test_{metric}"]
            row[metric] = f"{values.mean():.3f} ± {values.std():.3f}"
            row[f"_{metric}"] = values.mean()
        rows.append(row)
        print(f"{feature_set:16s} {name:22s} F1 = {row['f1']}")

results = pd.DataFrame(rows)
display(results[["variables", "modele", "f1", "rappel", "precision", "auc"]])

In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(x="_f1", y="modele", hue="variables", data=results)
plt.xlabel("F1 « achat » en validation croisée")
plt.ylabel("")
plt.title("Avec ou sans PageValues")
plt.show()

### 4.1 bis Comparer chaque modèle à son meilleur seuil

Le F1 de la section 4.1 est calculé au seuil de 0,5 : un modèle qui donne des probabilités prudentes (comme la forêt aléatoire) prédit alors presque jamais « achat » et paraît mauvais, alors qu'il **classe** peut-être bien les sessions (voir son AUC). Comme la partie 5 choisira de toute façon le seuil, il est plus juste de comparer les modèles **chacun à son meilleur seuil**.

Tout se fait en validation croisée sur l'entraînement (`cross_val_predict`) : **le test n'est toujours pas utilisé**. On ajoute aussi la précision moyenne (`average_precision`), un score qui ne dépend d'aucun seuil et qui convient aux classes déséquilibrées.

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import cross_val_predict

rows_seuil = []
for feature_set in FEATURE_SETS:
    for name in MODELS[1:]:   # la référence naïve n'a pas de probabilités utiles
        proba = cross_val_predict(make_pipeline(name, feature_set), X_train, y_train,
                                  cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
        precision, recall, thresholds = precision_recall_curve(y_train, proba)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        i = int(np.argmax(f1[:-1]))
        rows_seuil.append({"variables": feature_set, "modele": name,
                           "seuil": round(float(thresholds[i]), 3),
                           "f1_seuil": round(float(f1[i]), 3),
                           "rappel_seuil": round(float(recall[i]), 3),
                           "precision_seuil": round(float(precision[i]), 3),
                           "precision_moyenne": round(average_precision_score(y_train, proba), 3)})

results_seuil = pd.DataFrame(rows_seuil)
results = results.merge(results_seuil[["variables", "modele", "f1_seuil"]], on=["variables", "modele"], how="left")
display(results[["variables", "modele", "f1", "f1_seuil"]].rename(
    columns={"f1": "F1 au seuil 0,5 (moy. ± écart-type)", "f1_seuil": "F1 au meilleur seuil"}))
display(results_seuil)

plot = results.dropna(subset=["f1_seuil"]).melt(id_vars=["variables", "modele"], value_vars=["_f1", "f1_seuil"],
                                                var_name="mesure", value_name="F1")
plot["mesure"] = plot["mesure"].map({"_f1": "seuil 0,5", "f1_seuil": "meilleur seuil"})
grid = sns.catplot(x="F1", y="modele", hue="mesure", col="variables", data=plot, kind="bar", height=3.5, aspect=1.4)
grid.set_ylabels("")
plt.show()

### 4.2 Lecture des résultats (à rédiger)

- *Écart avec / sans `PageValues` :* …
- *Ce que cela dit sur la fuite possible :* …
- *Seuil 0,5 ou meilleur seuil (4.1 bis) : le classement des modèles change-t-il ?* …
- *Modèle le plus prometteur pour chaque variante :* …

### 4.3 Réglage des hyperparamètres avec `GridSearchCV`

Pour chaque variante, on règle le modèle qui a le meilleur F1 **à son meilleur seuil** (4.1 bis). Le réglage est noté avec la précision moyenne (`average_precision`), qui ne dépend pas du seuil : le seuil final sera choisi en partie 5.

In [ ]:
from sklearn.model_selection import GridSearchCV

PARAM_GRIDS = {
    "Regression_logistique": {"model__C": [0.01, 0.1, 1, 10]},
    "Foret_aleatoire": {"model__max_depth": [None, 10, 20], "model__min_samples_leaf": [1, 5]},
    "Gradient_boosting": {"model__learning_rate": [0.05, 0.1], "model__max_depth": [None, 6]},
}

best = {}
for feature_set in FEATURE_SETS:
    subset = results[(results.variables == feature_set) & (results.modele != "Reference_naive")]
    name = subset.sort_values("f1_seuil", ascending=False).iloc[0]["modele"]
    search = GridSearchCV(make_pipeline(name, feature_set), PARAM_GRIDS[name],
                          scoring="average_precision", cv=CV, n_jobs=-1)
    search.fit(X_train, y_train)
    params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
    best[feature_set] = {"model": name, "params": params, "cv_precision_moyenne": round(float(search.best_score_), 3)}
    print(f"{feature_set:16s} {name:22s} précision moyenne CV = {search.best_score_:.3f} | {params}")

### 4.4 Configuration retenue

Le dictionnaire `best` ci-dessous est utilisé **directement** dans la partie 5 : il n'y a rien à recopier à la main.

In [ ]:
print("BEST =", best)

## Bilan J2 (à rédiger)

- *Modèle retenu pour chaque variante et pourquoi :* …
- *Hyperparamètres trouvés :* …

# Partie 5 · Évaluation (étape 5)

### 5.0 Configuration retenue en partie 4

`BEST` reprend automatiquement les modèles et réglages trouvés en partie 4 (aucune valeur écrite à la main).

In [ ]:
BEST = best
DEPLOYED = "sans_PageValues"   # la seule variante entièrement connue pendant la visite
print("BEST =", BEST)

### 5.1 Choix du seuil en validation croisée (sans toucher au test)

In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import cross_val_predict


def choose_threshold(feature_set):
    cfg = BEST[feature_set]
    pipe = make_pipeline(cfg["model"], feature_set, cfg["params"])
    proba = cross_val_predict(pipe, X_train, y_train, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_train, proba)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    i = int(np.argmax(f1[:-1]))
    return float(thresholds[i]), precision, recall, f1


THRESHOLDS = {}
for feature_set in FEATURE_SETS:
    thr, precision, recall, f1 = choose_threshold(feature_set)
    THRESHOLDS[feature_set] = thr
    print(f"{feature_set:16s} seuil = {thr:.3f} | F1 CV au seuil = {f1.max():.3f}")
    plt.plot(recall, precision, label=feature_set)
plt.xlabel("Rappel")
plt.ylabel("Précision")
plt.title("Courbe précision-rappel (validation croisée sur le train)")
plt.legend()
plt.show()

### 5.2 Évaluation unique sur le test

In [ ]:
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, classification_report,
                             f1_score, recall_score, roc_auc_score)

FINAL = {}
metrics = []
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, feature_set in zip(axes, FEATURE_SETS):
    cfg = BEST[feature_set]
    pipe = make_pipeline(cfg["model"], feature_set, cfg["params"]).fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    y_pred = (proba >= THRESHOLDS[feature_set]).astype(int)
    FINAL[feature_set] = {"pipe": pipe, "y_pred": y_pred, "proba": proba}
    metrics.append({"variables": feature_set, "modele": cfg["model"],
                    "seuil": round(THRESHOLDS[feature_set], 3),
                    "f1": round(f1_score(y_test, y_pred), 3),
                    "rappel": round(recall_score(y_test, y_pred), 3),
                    "auc": round(roc_auc_score(y_test, proba), 3)})
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, display_labels=["Pas d'achat", "Achat"])
    ax.set_title(feature_set)
plt.tight_layout()
plt.show()

metrics = pd.DataFrame(metrics)
display(metrics)
print(classification_report(y_test, FINAL[DEPLOYED]["y_pred"], target_names=["Pas d'achat", "Achat"]))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for feature_set in FEATURE_SETS:
    RocCurveDisplay.from_predictions(y_test, FINAL[feature_set]["proba"], name=feature_set, ax=ax)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title("Courbes ROC sur le test")
plt.show()

### 5.3 Où le modèle se trompe-t-il ?

Rappel (part des acheteurs détectés) par groupe, pour le modèle déployé.

In [ ]:
errors = X_test.assign(achat=y_test.values, detecte=FINAL[DEPLOYED]["y_pred"])
buyers = errors[errors.achat == 1]
for col in ["VisitorType", "Month"]:
    table = buyers.groupby(col)["detecte"].agg(rappel="mean", acheteurs="size").round(3)
    display(table.sort_values("rappel"))

### 5.4 Limites (à rédiger, puis reporter dans la section 4 du README)

- *Erreurs les plus fréquentes :* …
- *Groupes mal servis :* …
- *Usages à éviter :* …

# Partie 6 · Déploiement (étape 6)

### 6.1 Export pour l'application

In [ ]:
import joblib

deployed_cols = FEATURE_SETS[DEPLOYED]
num_cols = [c for c in deployed_cols if c in NUM_COLS + [LEAKY_COL]]
bundle = {
    "pipeline": FINAL[DEPLOYED]["pipe"],
    "threshold": THRESHOLDS[DEPLOYED],
    "feature_set": DEPLOYED,
    "features": deployed_cols,
    "num_cols": num_cols,
    "cat_cols": CAT_COLS,
    "bool_cols": BOOL_COLS,
    # bornes vues à l'entraînement : l'application refuse ce qui sort de ce domaine
    "schema": {
        "num": {c: [float(X_train[c].min()), float(X_train[c].max())] for c in num_cols},
        "cat": {c: sorted(X_train[c].unique().tolist()) for c in CAT_COLS},
    },
    "metrics_test": metrics[metrics.variables == DEPLOYED].iloc[0].to_dict(),
    "versions": {"scikit-learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__},
}
joblib.dump(bundle, "model.joblib")

# requirements.txt avec les versions EXACTES de cet environnement (évite les erreurs de chargement)
Path("requirements.txt").write_text(
    "streamlit\n"
    f"scikit-learn=={sklearn.__version__}\n"
    f"pandas=={pd.__version__}\n"
    f"numpy=={np.__version__}\n"
    f"joblib=={joblib.__version__}\n"
)
print(Path("requirements.txt").read_text())
print("Modèle exporté :", bundle["metrics_test"])

try:
    from google.colab import files
    files.download("model.joblib")
    files.download("requirements.txt")
except ImportError:
    print("Hors Colab : fichiers écrits dans", Path.cwd())

## Bilan J3 (à rédiger)

- *Modèle déployé, seuil et scores sur le test :* …
- *Pourquoi la variante sans `PageValues` :* …
- Copiez `model.joblib` et `requirements.txt` dans le dossier `app/`, puis poussez-le sur GitHub.